In [1]:
import onnxruntime as ort
from jetbot import Robot, Camera
import numpy as np
import ipywidgets.widgets as widgets
from jetbot import bgr8_to_jpeg
import traitlets
from time import sleep

class Model():
    def __init__(self, model_file):
        self.model = ort.InferenceSession(model_file, providers=['CPUExecutionProvider'])
        self.camera = Camera()
        
        #self.robot = Robot()
    
    def run(self, widget=False):
        image = np.frombuffer(self.camera.value, dtype=np.uint8)
        image = image.reshape((3, self.camera.height, self.camera.width,1)).astype(np.float32) / 255
        input_name = self.model.get_inputs()[0].name
        if widget:
            image1 = widgets.Image(format='jpeg', width=500, height=500)
            camera_link = traitlets.dlink((self.camera, 'value'), (image1, 'value'), transform=bgr8_to_jpeg)
            display(image1)
        res = self.model.run(None, {input_name: image})
        print(res)
        return res

In [2]:
class Driver():
    def __init__(self, model_path, refresh_delay=.1):
        self.robot = Robot()
        self.model = Model(model_path)
        self.refresh_delay = refresh_delay
    
    def set_movement(self, x, y, dbg=False):
        # Negative is fwd, left
        if dbg:
            print('Move', x, y)
        self.robot.left_motor.value  = -.5 * y + .1 * x
        self.robot.right_motor.value = -.5 * y - .1 * x
        
    def run_loop(self):
        keep_going = True
        while keep_going:
            try:
                x, y = self.model.run()
                self.set_movement(x, y)
                sleep(self.refresh_delay)
            except KeyboardInterrupt:
                keep_going = False
                self.set_movement(0, 0)

In [3]:
d = Driver('./model_final_final_02_final_please.onnx')

In [4]:
d.run_loop()